# 15 — Production Engineering, Security, Optimization, Deployment & Capstone

Đây là graduation notebook. Không chỉ “agent trả lời được”.

## Learning requirements
- guardrails;
- prompt injection / indirect prompt injection;
- authorization and least privilege;
- side-effect approval + idempotency;
- async/concurrency/timeouts/retry/backoff;
- cost/latency measurement;
- deployment/persistence/secrets;
- observability/evaluation release gate.

## Threat model checklist

Tối thiểu xem xét:
1. prompt injection;
2. indirect injection từ RAG/web/MCP;
3. tool poisoning/misleading descriptions;
4. excessive agency;
5. cross-tenant memory leakage;
6. secret exposure;
7. unbounded loops/cost;
8. destructive tool call;
9. duplicate side effects after retry;
10. unsafe generated code/file operations.

Defense không chỉ bằng system prompt. Dùng deterministic authorization/policy boundaries.

In [ ]:
# Deterministic authorization example.
from dataclasses import dataclass

@dataclass(frozen=True)
class RequestContext:
    user_id: str
    role: str

def authorize(ctx: RequestContext, action: str) -> bool:
    policy = {
        "viewer": {"read_project"},
        "admin": {"read_project", "send_email", "delete_project"},
    }
    return action in policy.get(ctx.role, set())

ctx = RequestContext(user_id="u1", role="viewer")
assert authorize(ctx, "read_project")
assert not authorize(ctx, "delete_project")

## Async/concurrency

Production agent thường I/O-bound.

Học:
- `ainvoke`, `astream`;
- async tools;
- parallel independent work;
- bounded concurrency;
- timeout;
- retry with backoff;
- cancellation;
- rate limits.

Parallel không đồng nghĩa “càng nhiều càng nhanh”: provider quota và duplicated context có thể làm cost/latency tệ hơn.

## Optimization metrics

Đo trước khi optimize:
- TTFT;
- total latency;
- input/output tokens;
- tool latency;
- retrieval latency;
- model/tool call count;
- cost per successful task;
- failure/retry rate.

Techniques:
- dynamic model routing;
- trim/summarize context;
- offload large artifacts;
- prompt/provider caching;
- parallelize only independent operations;
- smaller model for simple stages;
- better retrieval;
- fewer irrelevant tools.

# Capstone — AI Interview Agent / Codebase Interview System

## Target architecture

```text
Wizard: project name + project goal
          |
          v
   Source Code Scanner
          |
          v
   Project Understanding
          |
          v
   Interview Supervisor
      /      |       \
     /       |        \
  Scanner  Analyst   Question Specialist
     \       |        /
      \      |       /
       Interview State
             |
             v
    Ask user / choices
             |
      need approval?
        /       \
      yes       no
       |         |
    interrupt    |
       \        /
        validate
           |
           v
    Final Specification
```

## Functional requirements
- initial wizard chỉ hỏi project name + initial description;
- scan source code để tạo project overview;
- identify unknowns/ambiguities;
- interview user bằng free-text hoặc suggested choices;
- không hỏi lại fact đã biết từ source/state;
- persist interview state;
- resume after refresh/restart;
- human control với sensitive/destructive actions;
- produce structured final specification.

## Technology requirements

### LangChain
- models/messages;
- tools;
- structured output;
- `create_agent`;
- middleware;
- streaming/context engineering.

### LangGraph
- explicit state;
- conditional edges;
- persistence;
- interrupt/resume;
- subgraphs;
- durable execution.

### RAG
- source chunks;
- metadata;
- retrieval evaluation;
- citations/evidence.

### Multi-agent / Deep Agents
Chỉ dùng nơi benchmark chứng minh specialization/context isolation có lợi.

### MCP
Có thể expose Git/repository/project tools qua MCP boundary.

### LangSmith
Trace + evaluation dataset + experiment comparison.

## Required repository artifacts

```text
artifacts/capstone/
|- architecture.md
|- state-schema.md
|- tool-catalog.md
|- context-boundaries.md
|- eval-dataset.jsonl
|- evaluation-results.md
|- security-threat-model.md
`- runbook.md
```

## Production acceptance criteria

Không graduate nếu chỉ có happy path.

Tối thiểu test:
- source unavailable;
- unsupported repository;
- tool timeout;
- invalid structured output;
- model rate limit;
- repeated question prevention;
- thread resume;
- user A/B isolation;
- prompt injection in source file;
- MCP tool failure;
- rejected approval;
- crash + resume;
- duplicate side-effect prevention.

## Final done criteria

Bạn phải có thể defend architecture decisions:
- tại sao dùng/không dùng agent;
- tại sao LangChain vs LangGraph vs Deep Agents ở từng layer;
- tại sao single/multi-agent;
- context nào được thấy;
- state/memory scopes;
- quality metrics;
- security boundaries;
- latency/cost trade-offs.